<a href="https://colab.research.google.com/github/mdzubayereashraf/Ostad-AI-Engineering-Course-Assignments/blob/main/Assignmen_No_19(Fine_Tuning_Transformers_for_Question_Answering_%7C%7C_Text_Classification_%26_Sentiment_Analysis_using_Hugging_Face).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction**

Natural Language Processing (NLP) is a branch of Artificial Intelligence that enables computers to understand, interpret, and process human language. In this assignment, I will fine-tune a pre-trained BERT (Bidirectional Encoder Representations from Transformers) model for the Question Answering (QA) task using the Stanford Question Answering Dataset (SQuAD v1.1) and the Hugging Face Transformers library.

### **Text Classification vs. Question Answering:**

Text Classification and Question Answering are two important NLP tasks, but they serve different purposes.

- **Text Classification:** It involves assigning predefined categories or labels to a given text. For example, classifying a movie review as positive or negative is a sentiment analysis task.

- **Question Answering:** It involves finding a relevant answer to a question based on a given context. For example, if the context describes Albert Einstein's achievements and the question asks who developed the theory of relativity, the model should identify "Albert Einstein" as the answer.

### **Objective:**

The main objective of this assignment is to fine-tune the pre-trained BERT model on the SQuAD v1.1 dataset so that it can understand questions and identify relevant answers from given passages.

I will use the Hugging Face Transformers library for tokenization and model training, and evaluate the model's performance using Exact Match (EM) and F1 Score.

### **Install and import libraries**

In [1]:
!pip install transformers datasets evaluate accelerate

In [2]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

import evaluate

### **Load the SQuAD Dataset**

In [4]:
dataset = load_dataset("rajpurkar/squad")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

### **Explore Dataset**

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [6]:
print(dataset["train"][0])

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [7]:
sample = dataset["train"][0]

print("Question:", sample["question"])

print("\nContext:", sample["context"])

print("\nAnswer:", sample["answers"])

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Answer: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


### **Load BERT Tokenizer**

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
question = "Who developed the theory of relativity?"

context = "Albert Einstein developed the theory of relativity in the early 20th century."

inputs = tokenizer(question, context)

print(inputs)

{'input_ids': [101, 2040, 2764, 1996, 3399, 1997, 20805, 1029, 102, 4789, 15313, 2764, 1996, 3399, 1997, 20805, 1999, 1996, 2220, 3983, 2301, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [10]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])

print(tokens)

['[CLS]', 'who', 'developed', 'the', 'theory', 'of', 'relativity', '?', '[SEP]', 'albert', 'einstein', 'developed', 'the', 'theory', 'of', 'relativity', 'in', 'the', 'early', '20th', 'century', '.', '[SEP]']


### **Tokenization and Preprocessing**

In [11]:
max_length = 384
stride = 128

In [15]:
def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_mapping = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        sample_index = sample_mapping[i]

        answer = examples["answers"][sample_index]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        cls_index = inputs["input_ids"][i].index(
            tokenizer.cls_token_id
        )

        # Find the context token boundaries

        token_start_index = 0

        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(sequence_ids) - 1

        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Check whether the answer is inside this chunk

        if (
            offsets[token_start_index][0] > start_char
            or offsets[token_end_index][1] < end_char
        ):

            start_positions.append(cls_index)
            end_positions.append(cls_index)

        else:

            # Find the starting token

            while (
                token_start_index <= token_end_index
                and offsets[token_start_index][0] <= start_char
            ):
                token_start_index += 1

            start_positions.append(token_start_index - 1)

            # Find the ending token

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1

            end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [16]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [17]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 88524
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 10784
    })
})


### **Load Pretrained BERT Model**

In [18]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-uncased"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [19]:
print(model)

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=